Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from uszipcode import SearchEngine
import requests
from bs4 import BeautifulSoup
from io import StringIO
from ast import literal_eval 

from pathlib import Path
import os

Downloaded CSV Dataset

In [51]:
def csv_parser(csv_file):
  df = pd.read_csv(csv_file)
  df = df.drop(columns=["LATITUDE","LONGITUDE","ON STREET NAME","CROSS STREET NAME",
                        "OFF STREET NAME","NUMBER OF PERSONS INJURED","NUMBER OF PERSONS KILLED",
                        "NUMBER OF CYCLIST INJURED","NUMBER OF CYCLIST KILLED","NUMBER OF MOTORIST INJURED",
                        "NUMBER OF MOTORIST KILLED","COLLISION_ID"])
  
  df = df.iloc[:,:7]
  
  df["ZIP CODE"] = df["ZIP CODE"].fillna(0)
  df["ZIP CODE"] = df["ZIP CODE"].astype(str).str.strip().replace("",0)
  df["ZIP CODE"] = df["ZIP CODE"].astype(float).astype(int).astype(str)

  df["LOCATION"] = df["LOCATION"].fillna("(0, 0)")
  df["LOCATION"] = df["LOCATION"].str.strip()
  
  df = df[(df["ZIP CODE"] != "0") | (df["LOCATION"] != "(0, 0)")]

  df["BOROUGH"] = df["BOROUGH"].fillna("UNKNOWN")
  df["CRASH DATE"] = pd.to_datetime(df["CRASH DATE"]).dt.year

  df = df.rename(columns={"CRASH DATE":"CRASH YEAR"})

  df = df.dropna()

  df["NUMBER OF PEDESTRIANS AFFECTED"] = df["NUMBER OF PEDESTRIANS INJURED"] + df["NUMBER OF PEDESTRIANS KILLED"]

  current_dir = Path(os.getcwd()).resolve()
  if current_dir.name in ["notebooks", "src"]:
      project_root = current_dir.parent
  else:
      project_root = current_dir
  CLEANED_FOLDER = project_root / "data" / "cleaned"
  OUTPUT_FILE_PATH = CLEANED_FOLDER / "cleaned_accident_data.csv"
  CLEANED_FOLDER.mkdir(parents=True, exist_ok=True)
  df.to_csv(OUTPUT_FILE_PATH, index=False)
  
  # df.to_csv("data/cleaned/csv_output.csv", index=False)

  return OUTPUT_FILE_PATH

def get_zip_code(df):
  unique_locations = df[df["ZIP CODE"] == 0]["LOCATION"].unique() # gets all of the unique latitute/longitude coordinates since there are repeats

  search = SearchEngine() # SearchEngine object used to run the by_coordinates method
  locationDict = {}

  for location in unique_locations:
      latitude_longitude = location.strip('()').split(',')
      latitude, longitude = float(latitude_longitude[0]), float(latitude_longitude[1])
      
      result = search.by_coordinates(latitude, longitude, radius=1, returns=1) # by_coordinates method lets you find a zip code by latitude/longitude within a certain radius (in this case 1)

      if result:
          locationDict[location] = result[0].zipcode # gets the zipcode value from the result of the by_coordinates method
      else: 
          locationDict[location] = 0

  df[df["ZIP CODE"] == 0]["ZIP CODE"] = df[df["ZIP CODE"] == 0]["LOCATION"].map(locationDict) # uses the Pandas .map method to map the calculated zip codes to the right locations in the DataFrame
  df["ZIP CODE"] = df["ZIP CODE"].fillna(0)

  df = df[df["ZIP CODE"] != 0]

  current_dir = Path(os.getcwd()).resolve()
  if current_dir.name in ["notebooks", "src"]:
      project_root = current_dir.parent
  else:
      project_root = current_dir
  CLEANED_FOLDER = project_root / "data" / "cleaned"
  OUTPUT_FILE_PATH = CLEANED_FOLDER / "final_cleaned_accident_data.csv"
  CLEANED_FOLDER.mkdir(parents=True, exist_ok=True)
  df.to_csv(OUTPUT_FILE_PATH, index=False)

#   df.to_csv("data/cleaned/final_csv_output.csv", index=False)

  return df

API Dataset

In [74]:
def json_api_parser(url):
  api_key = "5771b86a16a9999b827c3d2ef7dda48b07ab7815"
  data = requests.get(url + api_key).json()
  
  df = pd.DataFrame(data = data[1:], columns = data[0])
  df = df.drop(["NAME"], axis = 1)
  df = df.rename(columns = {"zip code tabulation area" : "zip_code", "B19013_001E" : "median_household_income", "B08301_019E" : "walk_to_work", "B08301_010E" : "public_transport_work", "B01003_001E" : "total_population"})
  df = df[df["zip_code"].str.match(r"^1(0[0-9]|1[0-6])\d{2}$")]
#   df = df.set_index("zip_code")

  current_dir = Path(os.getcwd()).resolve()
  if current_dir.name in ["notebooks", "src"]:
      project_root = current_dir.parent
  else:
      project_root = current_dir
  CLEANED_FOLDER = project_root / "data" / "cleaned"
  OUTPUT_FILE_PATH = CLEANED_FOLDER / "final_cleaned_api_data.csv"
  CLEANED_FOLDER.mkdir(parents=True, exist_ok=True)
  df.to_csv(OUTPUT_FILE_PATH, index=False)

  # df.to_csv("data/cleaned/final_api_output.csv", index = True)
  return df

Webscraped Dataset

In [56]:
def find_match(scraped_name, new_map):
    for map_name in new_map.index:
        if scraped_name in map_name:
            return new_map[map_name]
    return "[]"

def html_parser(url):
    current_dir = Path.cwd()
    while current_dir.name != "nyc-walkability-accident-analysis" and current_dir != current_dir.parent:
        current_dir = current_dir.parent

    DATA_PATH = current_dir / "data" / "raw" / "NeighborhoodToZipCode.csv"

    map_df = pd.read_csv(DATA_PATH)

    map_df['UHFNAME'] = map_df['UHFNAME'].str.strip().str.upper()
    map_df = map_df.rename(columns={"UHFNAME":"Neighborhood"})

    new_map = map_df.groupby("Neighborhood")["ZCTA"].apply(list)

    response = requests.get(url)
    soup = BeautifulSoup(response.text, 'html.parser')
    
    table = soup.find("table",{"id":"hoods-list-table"})
    rows = table.find("tbody").find_all("tr")

    data = []
    for row in rows:
        cols = row.find_all("td")
        if len(cols) > 0:
            name = cols[1].text.strip()
            walk_score = cols[2].text.strip()

            data.append({
                "Name": name,
                "Walk Score": walk_score
            })

    df = pd.DataFrame(data)
    
    df["Name"] = df["Name"].str.strip().str.upper()
    df['Zip Codes'] = df['Name'].apply(find_match, args=(new_map,))

    current_dir = Path(os.getcwd()).resolve()
    if current_dir.name in ["notebooks", "src"]:
        project_root = current_dir.parent
    else:
        project_root = current_dir
    CLEANED_FOLDER = project_root / "data" / "cleaned"
    OUTPUT_FILE_PATH = CLEANED_FOLDER / "final_cleaned_web_data.csv"
    CLEANED_FOLDER.mkdir(parents=True, exist_ok=True)
    df.to_csv(OUTPUT_FILE_PATH, index=False)
        
    # df.to_csv("data/cleaned/final_web_output.csv", index=False)
    return df

Function Calls

Downloaded Data

In [53]:
current_dir = Path.cwd()
while current_dir.name != "nyc-walkability-accident-analysis" and current_dir != current_dir.parent:
    current_dir = current_dir.parent

DATA_PATH = current_dir / "data" / "raw" / "accident_data.csv"

# Downloaded data
output1 = csv_parser(DATA_PATH)
df = pd.read_csv(output1)
df_downloaded = get_zip_code(df)

/var/folders/xs/r1jtw3y5665395typ4r_12g00000gn/T/ipykernel_3581/533145977.py:2: DtypeWarning: Columns (3) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(csv_file)
/var/folders/xs/r1jtw3y5665395typ4r_12g00000gn/T/ipykernel_3581/533145977.py:59: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df[df["ZIP CODE"] == 0]["ZIP CODE"] = df[df["ZIP CODE"] == 0]["LOCATION"].map(locationDict) # uses the Pandas .map method to map the calculated zip codes to the right locations in the DataFrame


API Data

In [75]:
# API data
df_api = json_api_parser("https://api.census.gov/data/2024/acs/acs5?get=NAME,B19013_001E,B08301_019E,B08301_010E,B01003_001E&for=zip%20code%20tabulation%20area:*&key=")


Webscraped Data

In [67]:
# Webscraped data
df_web = html_parser("https://www.walkscore.com/NY/New_York")

In [ ]:
from ast import literal_eval 

def join_data():
    # df_crash = pd.read_csv("final_csv_output.csv")
    # df_census = pd.read_csv("final_api_output.csv")
    # df_walkability = pd.read_csv("final_web_output.csv")

    df_walkability = df_web

    df_walkability['Zip Codes'] = df_walkability['Zip Codes'].apply(
        lambda x: literal_eval(x) if isinstance(x, str) else x
    )

    # df_walkability['Zip Codes'] = df_walkability['Zip Codes'].apply(literal_eval) 
    df_walkability = df_walkability.explode('Zip Codes').rename(columns={'Zip Codes': 'zip_code'}).dropna() 
    
    df_downloaded['ZIP CODE'] = df_downloaded['ZIP CODE'].astype(str)
    df_api['zip_code'] = df_api['zip_code'].astype(str)
    df_walkability['zip_code'] = df_walkability['zip_code'].astype(str)

    df = pd.merge(df_downloaded, df_api, left_on='ZIP CODE', right_on='zip_code', how='inner')
    df = pd.merge(df, df_walkability, on='zip_code', how='left')
    
    df['median_household_income'] = pd.to_numeric(df['median_household_income'], errors='coerce')
    df = df[df['median_household_income'] > 0]

    return df

df = join_data()

current_dir = Path(os.getcwd()).resolve()
if current_dir.name in ["notebooks", "src"]:
    project_root = current_dir.parent
else:
    project_root = current_dir
CLEANED_FOLDER = project_root / "data" / "cleaned"
OUTPUT_FILE_PATH = CLEANED_FOLDER / "final_data.csv"
CLEANED_FOLDER.mkdir(parents=True, exist_ok=True)
df.to_csv(OUTPUT_FILE_PATH, index=False)

# df.to_csv("final_csv.csv", index=False)

/var/folders/xs/r1jtw3y5665395typ4r_12g00000gn/T/ipykernel_3581/2607794692.py:17: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_downloaded['ZIP CODE'] = df_downloaded['ZIP CODE'].astype(str)
